# R07 · Cookbook-Referenzlabor: Beispiele, Einschränkungen und API-Referenz

**NHyDB-0.28.2-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb-run4/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitte 25–27), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine Zelle einzeln aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel steht die im Cookbook abgedruckte Ausgabe, zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (Nachschlagetext, keine ausführbaren Beispiele)

* 25. Vollständige Anwendungsbeispiele
* 26. Bekannte Einschränkungen
* 27. API-Referenz

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.28.2':
    print('⚠ These examples were written for NHyDB 0.28.2; results may differ.')

nhydb 0.28.2 · file format 0.28.2


## 25. Vollständige Anwendungsbeispiele

Ausführbare Programme in [`examples/python/`](https://github.com/nhydb-run4/NHyDB/blob/main/examples/python); jedes
legt seine Datenbank in einem temporären Verzeichnis an:

| Programm | Zeigt |
|---|---|
| [`sensor_timeseries.py`](https://github.com/nhydb-run4/NHyDB/blob/main/examples/python/sensor_timeseries.py) | NumPy-Batch-Schreiben eines Tages Sensordaten, Lesen von Zeitfenstern mit `read_numpy`, NHySQL-Aggregation (braucht NumPy) |
| [`document_store.py`](https://github.com/nhydb-run4/NHyDB/blob/main/examples/python/document_store.py) | Datensätze mit Extensions, VALUE- und FULLTEXT-Indizes, BM25-Suche, Phrasenabfragen, NHySQL-Gruppierung |
| [`math_map_grid.py`](https://github.com/nhydb-run4/NHyDB/blob/main/examples/python/math_map_grid.py) | MATH_MAP-Planung mit `dry_run`, Schreiben und Lesen von Zellen über Eingabe-Tupel, Suche über die QueryEngine |

## 26. Bekannte Einschränkungen

* **Ein schreibendes Objekt pro Datenbank.** Die exklusive Sperre wird bei
  jedem Öffnen zum Lesen und Schreiben gesetzt; weitere Leser verwenden `readonly=True` (keine
  Sperre, Zustand zum Zeitpunkt des Öffnens). Es gibt keine prozessübergreifenden Transaktionen.
* **Ein schreibender Aufruf pro Objekt.** Ein Schreiben auf ein Objekt, das ein anderer
  Thread gerade verwendet, löst `RuntimeError` aus; parallele nur lesende Abfragen auf einem
  Objekt sind erlaubt, eigene Objekte pro Thread skalieren am besten.
* **Rollback pro Objekt.** `rollback()` kehrt zum letzten Commit des
  Objekts zurück; Aufrufe über mehrere Blöcke sind für sich nicht atomar (Ausnahme:
  `put_math_batch`). Es gibt keine verschachtelten oder objektübergreifenden Transaktionen.
* **`batch()`** sieht innerhalb des Blocks die eigenen Schreibvorgänge nicht (kein read-your-writes).
* **Der Kontextmanager committet** beim Verlassen, auch nach einer Exception.
* **Schlüsseldatei.** Ohne die passende `*.layout.key` lässt sich die Datenbank nicht
  öffnen; ein verlorener Schlüssel lässt sich nicht wiederherstellen. Verschlüsselt ist nur das
  Schema, nicht die Daten.
* **Umbenannte Datenbankdateien** lassen sich nicht öffnen (nimm `compact_file(output=...)`).
* **Kein Upgrade älterer Dateien.** Datenbanken, die mit NHyDB 0.28.0,
  0.27.1 oder älter angelegt wurden, lösen `CorruptionError` aus; exportiere sie mit der Version,
  die sie geschrieben hat, und importiere die Daten neu (Abschnitt 21).
* **Indizes** werden einmal mit `build_indexes()` aufgebaut und danach automatisch
  gepflegt (`[index] maintenance = "auto"`, Abschnitt 13); sie decken nur
  CLASSIC-Datensätze ab.
* **NHySQL** ist nur lesend und arbeitet auf CLASSIC-Layouts; kein `DISTINCT`,
  `HAVING` und keine Joins.
* **QueryEngine `order_by`** wirkt nur für Felder, die in
  `select` aufgeführt sind (0.28.0).
* **Große Batches:** Der erste Zugriff auf einen Block liest und prüft den
  ganzen Block einmal (ein Block mit 1 Million Datensätzen hat etwa 25 MB); spätere
  Zugriffe lesen nur die benötigten Bytes.
* **CUSTOM-Layouts** brauchen nach jedem Öffnen ihr registriertes Plugin;
  Plugins dürfen während eines Callbacks nicht in dieselbe Datenbank schreiben.
* **`[cache]`-Konfiguration:** `block_cache_bytes` begrenzt den Block-Cache;
  `vindex_max_marks`/`vindex_max_indexes` begrenzen die v-Indizes großer
  einzelner u-Gruppen (Abschnitt 19).
* **NumPy-Koordinaten-Arrays** (`u`, `v`, Objekt-IDs) sind auf
  `uint64` begrenzt; für größere Werte nimm Listen von Python-`int`.
* **Durability `full`** löst auf macOS-Dateisystemen ohne
  `F_FULLFSYNC`-Unterstützung einen Fehler aus.

## 27. API-Referenz

Abschnittsnummern beziehen sich auf die Rezepte oben. Parameter in *Kursivschrift* sind
nur als Schlüsselwort übergebbar (keyword-only). Alle Schreibmethoden akzeptieren *`commit=False`*; Lese- und
Schreibmethoden mit Lane akzeptieren *`lane=None`* (`NL`).

### 27.1 Modul `nhydb`

| Name | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `create(path, **options)` / `NHyDB.create` | Datenbank anlegen | *overwrite, durability, lock, lock_timeout_ms, layout_path, key_path, config* | 3, 4 |
| `open(path, **options)` / `NHyDB(path, ...)` | Datenbank öffnen | wie `create`, zusätzlich *readonly* (ohne *overwrite*) | 3, 4, 22 |
| `dynadron_n(realm, u, v, lane)` | Adresse → N | | 9 |
| `n_to_dynadron_address(n)` | N (int oder `"N…"`) → Adresse | | 9 |
| `parse_n_reference(ref)` | `"N589"` → 589 | | 9 |
| `cell_address(realm, u, v, lane)` | Adresse mit allen Kennzahlen | | 9 |
| `dynadron_address(realm, u, v)` | Kennzahlen ohne Lane | | 9 |
| `global_v`, `split_global_v`, `realm_v_bounds`, `validate_local_v` | Realm ↔ globales `v` | | 9 |
| `normalize_lane`, `lane_name` | Lane-Umwandlung | | 9 |
| `enc_vu`, `dec_vu`, `enc_vi`, `dec_vi` | VarUInt-/VarInt-Codecs | | 9 |
| `ENGINE_VERSION`, `FILE_FORMAT_VERSION`, `REALM_SPAN`, `TYPE_CODES` | Konstanten | | 1, 9, 11 |
| `LayoutDef`, `IndexDef`, `MathLayoutState` | von der API zurückgegebene Definitionsobjekte (`to_dict()`) | | 13, 17, 18 |
| `QueryEngine` | Abfrage-Engine (auch `nhydb.query.QueryEngine`) | | 14 |
| Exceptions | siehe die Hierarchie | | 22 |

### 27.2 `NHyDB` – Lebenszyklus, Attribute und Wartung

| Methode / Attribut | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `commit()` | Daten dauerhaft machen; liefert die Commit-Grenze | | 5 |
| `rollback()` | alle Änderungen seit dem letzten Commit verwerfen; liefert die Commit-Grenze | | 5 |
| `sync()` | Puffer schreiben (kein Commit) | *hard=False* | 6 |
| `close()` | Datei und Sperre freigeben | *commit=True* | 5 |
| `checkpoint()` | Checkpoint-Block schreiben und committen | | 20 |
| `closed`, `path`, `layout_path`, `key_path`, `config_source`, `readonly`, `open_warnings` | Zustand, Dateiorte (absolut) und Warnungen beim Öffnen | | 4, 21, 22 |
| `durability` | `"fsync"`/`"full"`, schreibbar | | 4, 5 |
| `max_db_size_bytes`, `set_max_db_size(size)` | Größengrenze | | 19 |
| `committed_end`, `last_valid_end`, `checkpoint_offset`, `next_block_id`, `recovered_tail_bytes` | Container-Zustand | | 4, 5, 6, 20 |
| `stats()`, `health()`, `verify()` | Statistik (inkl. `block_cache`) und Prüfungen | *deep=False* | 19, 20 |
| `NHyDB.inspect_tail(path)` | Datei prüfen (Superblöcke, Dateiende, beschädigte Blöcke), ohne sie zu öffnen | | 20 |
| `NHyDB.repair_file(path)` | einen konsistenten Zustand herstellen; schneidet committete Daten nie stillschweigend ab | *apply=False, make_backup=True, rebuild_indexes=True, allow_data_loss=False, keep_uncommitted=False*, Optionen für Begleitdateien | 20 |
| `NHyDB.restore_backup(backup, path)` | eine Sicherung von Reparatur/Kompaktierung zurückspielen | Optionen für Begleitdateien | 20 |
| `NHyDB.compact_file(path)` | an Ort und Stelle neu schreiben oder exportieren | *output=None, make_backup=True, rebuild_indexes=True, batch_records=10000*, Optionen für Begleitdateien | 20, 21 |
| `get_system_metadata()`, `get_system_cell(v)` | Systemraum R0,u1 | | 23 |

### 27.3 `NHyDB` – Schema und Layouts

| Methode / Attribut | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `create_realm(realm_id, name)` | Realm definieren (eine belegte ID löst `SchemaError` aus; ebenso bei Shapes, Layouts und Indizes) | *description, default_shape_id, base_reserve_to* | 3, 8 |
| `realms`, `realm_bounds(realm_id)` | Realm-Definitionen, `v`-Bereich | | 4, 9 |
| `create_shape(shape_id, realm_id, name, fields)` | BASE-Felder `[{v, name, type, enum?}]` | *description* | 3, 6 |
| `set_realm_default_shape(realm_id, shape_id)` | Standard-Shape | | 6 |
| `define_extension(realm_id, v, type)`, `define_extensions(realm_id, fields)` | Extension-Felder | *name, enum, description, replace* | 8 |
| `extension_catalog(realm_id)`, `resolve_v(realm_id, column)` | Extensions auflisten, Name → `v` | | 8 |
| `create_classic_layout(layout_id, realm_id, name, *, shape_id)` | CLASSIC-Layout | *base_reserve_to, u_start, u_end, priority* | 3, 6 |
| `create_column_layout(..., *, fields)` | COLUMN-Layout `[{u, name, type}]` | *anchor_u, priority* | 17 |
| `create_packed_layout(..., *, u, span, fields)` | PACKED-Layout `[{offset, name, type}]` | *priority* | 17 |
| `create_custom_layout(..., *, plugin, u_start, u_end)` | CUSTOM-Layout | *params, spacemap, priority* | 17 |
| `register_layout_plugin(name, plugin)` | Python-Plugin anhängen | | 17 |
| `create_layout(layout_id, realm_id, name, kind)` | allgemeine Form | *u_start, u_end, priority, shape_id, params, spacemap, plugin, replace* | 17 |
| `get_layout(...)`, `layout_at(realm, u)`, `layout_candidates(realm, u)`, `route_address(realm, u, v)`, `layout_diagnostics(realm)` | Routing | | 17 |
| `plan_layout_evolution(layout_id, ...)`, `evolve_layout(layout_id, ...)` | Layout sicher ändern | *name, description, u_start, u_end, priority, add_fields, spacemap_update* | 17 |
| `logical_address(layout_id, object_id, field)`, `record_anchor(layout_id, object_id)` | Objekt → Zelladresse | | 10, 17 |

### 27.4 `NHyDB` – Schreiben

| Methode | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `put_record(layout_id, object_id, values)` | ein Datensatz (dict) | | 3, 6 |
| `put(realm_id, u, values)` | ein Datensatz (Liste) | *shape_id, extensions* | 6, 8 |
| `put_dense_batch(realm_id, u_start, rows)` | aufeinanderfolgende Datensätze | *shape_id, extensions_by_u* | 6, 8 |
| `put_object(layout_id, object_id, values)`, `put_objects_batch(layout_id, objects)` | Datensätze beliebiger Layouts | *u_window* | 17 |
| `put_cell(realm_id, u, v, value)` | eine Zelle | *type* | 7 |
| `put_points(realm_id, cells)` | viele Zellen `(u, v, value[, type])` | *u_window* | 7 |
| `batch()` → `WriteBatch` | Kontextmanager, der `put_cell`/`delete_cell`/`delete_record`/`delete_area` sammelt und als eine Einheit schreibt; `flush()` (Zähler inkl. `deleted`), `discard()`, `len()` | *commit=False, u_window=10000* | 7 |
| `put_extensions(realm_id, u, mapping)`, `put_extension_list(realm_id, u, vs, values)`, `put_extension_range(realm_id, u, start_v, values)` | Extension-Werte | *merge=True* | 8 |
| `delete_record(layout_id, object_id)` | einen Datensatz vollständig löschen (jede lebende Zelle: Felder, einzelne Zellen, Extensions); liefert die Anzahl der gelöschten lebenden Zellen | | 6, 7 |
| `delete_area(realm_id, u_from, u_to, v_from, v_to)` | jede lebende Zelle eines Rechtecks löschen (Grenzen einschließlich), als eine Einheit; liefert die Anzahl der gelöschten lebenden Zellen | | 6, 7 |
| `delete_cell(realm_id, u, v)` | eine Zelle löschen; `True`, wenn sie lebte (ein gespeichertes `None` eingeschlossen) | | 7 |

### 27.5 `NHyDB` – Lesen und Adressen

| Methode | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `get_record(layout_id, object_id)`, `get_object(...)` | ein Datensatz | *include_addresses* (`get_object`) | 3, 6, 17 |
| `get(realm_id, u)` | dichter Datensatz mit Extensions | *include_extensions, dyn_address, dyn_v* | 6, 12 |
| `get_value(realm_id, u, v)` | Wert einer Zelle (`v` auch als Spaltenname) | | 3, 7 |
| `get_cell(realm_id, u, v)` | Zelle mit Metadaten | *dyn_address=True* | 7 |
| `get_internal(realm_id, u)` | Rohwerte nach `v` | | 6 |
| `get_record_view(layout_id, object_id)` | Datensatz mit allen Zellnummern | *request_n* | 10 |
| `range(realm_id, u_from, u_to)`, `select(realm_id, u_from, u_to, columns)`, `select_us(realm_id, us, columns)` | verzögerte (lazy) Iteratoren | *dyn_address, dyn_v, include_extensions* | 12 |
| `iter_materialized_cells`, `iter_materialized_sparse_points`, `iter_materialized_record_ns` | alles Gespeicherte | *realm_id* | 12 |
| `set_dyn_address_output(mode)` | Standard-Adresskennzahlen | *v* | 12 |
| `get_n(n)`, `get_n_cell(n)`, `get_n_record(n)`, `get_cell_n(n)`, `get_address(realm, u, v)` | Lesen über N / Adresse | *mode="auto"* | 10 |
| `describe_n(n)`, `resolve_n(n)` | ein N erklären | | 10 |
| `record_n_for_address(realm, u, v)`, `record_n_from_n(n)` | Anker-N des Datensatzes | | 10 |
| `occupied_n(layout_id, object_id)`, `occupied_from_n(n)` | belegte Zellen eines Objekts | *output="N"\|"ADDRESS"\|"BOTH"* | 10 |
| `cell_address(realm, u, v)`, `dyn_address(realm, u, v)` | Adresskennzahlen | | 10 |

### 27.6 `NHyDB` – Indizes, Abfragen, NumPy, MATH

| Methode / Attribut | Zweck | Wichtige Parameter | Siehe |
|---|---|---|---|
| `create_index(index_id, realm_id, name, field)` | VALUE-Index (oder andere) | *kind, options, tokenizer, casefold, ngram* | 13 |
| `create_token_index`, `create_trigram_index`, `create_fulltext_index` | Textindizes | *tokenizer, casefold, ngram* | 13 |
| `build_indexes(index_ids=None)` | aufbauen / aktualisieren | | 13 |
| `index_defs` | `{id: IndexDef}` | | 13 |
| `find_u(realm_id, filters)`, `find(realm_id, filters)` | Indexfilter → `u` / Datensätze | *u_from, u_to* | 13 |
| `posting_count(index_id, value)`, `index_candidates(index_id, op, value, value2)`, `index_candidate_estimate(...)` | direkter Indexzugriff | | 13 |
| `fulltext_term_posting`, `fulltext_corpus`, `fulltext_payloads` | Volltext-Interna | | 13 |
| `query(expr)`, `explain_query(expr)` | QueryEngine-Kurzformen | *realm, return_mode, select* | 14, 15 |
| `sql(text, params)`, `sql_numpy(text, params)`, `compile_sql(text, params)` | NHySQL | | 15, 16 |
| `put_numpy_batch(realm_id, u_start, columns)` | CLASSIC-Datensätze aus Arrays | *shape_id* | 16 |
| `put_objects_numpy(layout_id, object_ids, columns)` | Objekte beliebiger Layouts | *u_window* | 16 |
| `put_cells_numpy(realm_id, u, v, values)` | dünn besetzte Zellen | *type, u_window* | 16 |
| `read_numpy(realm_id, u_start, count, fields=None)` | Spalten + `_present` | *lane, object_fallback=False* | 16 |
| `create_math_layout(spec)`, `create_math_map_layout(name, parameters, *, u, V)` | MATH-Layouts | *layout_id, payload_type, side, realm_auto_limit* | 18 |
| `replace_math_spec(layout_id, spec)` | nur vor dem ersten Schreiben | *payload_type* | 18 |
| `put_math(layout_id, inputs, payload)`, `put_math_batch(layout_id, rows)`, `put_math_numpy(layout_id, inputs, payloads)` | MATH-Zellen schreiben | *overwrite=False* | 18 |
| `get_math(layout_id, inputs)`, `get_math_cell(layout_id, inputs)` | MATH-Zellen lesen | *dyn_address* | 18 |
| `describe_math(layout_id)`, `math_layouts` | MATH-Zustand | | 18 |

### 27.7 `nhydb.query`

| Name | Zweck | Siehe |
|---|---|---|
| `QueryEngine(db)` | `execute`, `explain`, `search`, `execute_spec`, `execute_numpy`, `compile_sql`, `execute_sql`, `execute_sql_numpy`, `register_math_provider`, `math_providers`, `db` | 14, 15, 16 |
| `Q.field`, `Q.u`, `Q.v`, `Q.realm`, `Q.lane`, `Q.N`, `Q.record_N`, `Q.struct`, `Q.n`, `Q.math`, `Q.fulltext`, `Q.phrase`, `Q.all` | Ausdrucks-Builder | 14 |
| `expr_from_spec(spec)`, `parse_sql(text)` | Ausdrücke aus Daten bauen / SQL parsen | 14, 15 |
| `QUERY_VERSION`, `SQL_LANGUAGE_VERSION` | Versionskennungen | 15 |

### 27.8 `nhydb.math`

| Name | Zweck | Siehe |
|---|---|---|
| `MathExpression.compile(text, parameters)` / `.evaluate(**values)` | sichere ganzzahlige Ausdrücke | 18 |
| `MathMap.compile(name, parameters, *, u, V, side)` | Abbildung; `target`, `targets`, `to_dict`, `from_dict`, `fingerprint` | 18 |
| `MathMapLayoutSpec(name, mapping, realm_auto_limit)` | Spezifikation für `create_math_layout` | 18 |
| `dry_run(mapping, rows)` → `MathDryRunReport` | Abbildung ohne Datenbank prüfen | 18 |
| `get_math_max_intermediate_bits()`, `set_math_max_intermediate_bits(bits)` | Ressourcengrenze (prozessweit, nur für eigenständige Objekte; Datenbanken verwenden den pro MATH-Layout eingefrorenen Konfigurationswert) | 18 |
| `split_global_v`, `join_global_v`, `normalize_side`, `NL`, `NR`, `REALM_SPAN`, `DEFAULT_*` | Hilfsfunktionen und Konstanten | 18 |

### 27.9 Bewusst nicht durch Rezepte abgedeckt

| Name | Grund |
|---|---|
| `nhydb.kernel`, `nhydb.index`, `nhydb.mathlayout` | zustandslose Bausteine der Engine (Codecs, Tokenizer, BM25-Kern, Realm-Planer); sie existieren für Konformitätstests und Werkzeuge. Verwende stattdessen die Datenbankmethoden oben. |
| `NHyDB.f` | nur lesende Low-Level-Dateiansicht (`seek`/`tell`), von Tests verwendet |
| Ausdrucks- und SQL-Klassen in `nhydb.query` (`FieldPredicate`, `BoolExpr`, `SQLCompiledQuery`, `QueryContext`, `CandidateSet`, …) | Datenklassen, die die API zurückgibt oder entgegennimmt; Ausdrücke baust du stattdessen mit `Q` |
| Attribute von `MathMapTarget`, `MathDryRunReport`, `MathLayoutState` | Ergebnisobjekte; ihre Felder werden dort gezeigt, wo sie zurückgegeben werden (Abschnitt 18) |
| `LayoutDef.contains_u`, `LayoutDef.to_dict`, `IndexDef.to_dict` | kleine Hilfsfunktionen an Ergebnisobjekten (Abschnitte 13, 17) |